In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,classification_report,confusion_matrix
print("libraries loaded succesfully")

libraries loaded succesfully


In [22]:
file4=r"C:\Users\RSC\Desktop\data sheets\project\data science project\customer_level_features.csv"
df = pd.read_csv(file4)
df.head()

,CustomerID,Recency,Frequency,Monetary,AverageOrderValue,PurchaseTrend,CustomerTenure,Churn
0,C00001,33,2,708.23,354.115,0.0,98,0
1,C00002,41,3,1705.44,568.480,1.0,130,0
2,C00004,96,1,199.66,199.660,-1.0,96,0
3,C00006,74,2,1152.75,576.375,0.0,221,0
4,C00009,42,1,54.88,54.880,NaN,42,1


In [76]:
df['PurchaseTrend'] = df['PurchaseTrend'].fillna(0)

In [77]:
print(df.dtypes)
print(df.shape)

CustomerID            object
Recency                int64
Frequency              int64
Monetary             float64
AverageOrderValue    float64
PurchaseTrend        float64
CustomerTenure         int64
Churn                  int64
dtype: object
(55, 8)


In [26]:
df['AverageOrderValue'] = df['AverageOrderValue'].astype(int)
df['Monetary'] = df['Monetary'].astype(int)
df['PurchaseTrend'] = df['PurchaseTrend'].astype(int)
df['CustomerID'] = df['CustomerID'].astype(str)

In [31]:
#Target and features

TARGET = "Churn"

feature_columns = [
    "AverageOrderValue",
    "Monetary",
    "PurchaseTrend"
]

x = df[feature_columns].copy()
y = df[TARGET].astype(int)

print(x.shape)
print(y.shape)

(55, 3)
(55,)


In [32]:
#Train-test-split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Samples:", x_train.shape[0])
print("Testing Samples:", x_test.shape[0])


Training Samples: 44
Testing Samples: 11


In [79]:
#preprocessing and features
numeric_features = feature_columns

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features)
    ]
)
preprocessor.fit(x_train,y_train)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['AverageOrderValue', 'Monetary',
                                  'PurchaseTrend'])])

In [63]:
def evaluate_model(model, x_test, y_test, model_name):

    y_pred = model.predict(x_test)

    print("\n" + "=" * 50)
    print(model_name)
    print("=" * 50)
       
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(
        precision_score(y_test, y_pred, zero_division=0), 4
    ))
    print("Recall:", round(
        recall_score(y_test, y_pred, zero_division=0), 4
    ))
    print("F1 Score:", round(
        f1_score(y_test, y_pred, zero_division=0), 4
    ))

    if hasattr(model, "predict_proba"):
        y_probability = model.predict_proba(x_test)[:, 1]

        print("ROC-AUC:", round(
            roc_auc_score(y_test, y_probability), 4
        ))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

In [67]:
#baseline model
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(
            strategy="most_frequent"
        ))
    ]
)

baseline_model.fit(x_train, y_train)

evaluate_model(
    baseline_model,
    x_test,
    y_test,
    "BASELINE MODEL - DUMMY CLASSIFIER"
)


BASELINE MODEL - DUMMY CLASSIFIER
Accuracy: 0.4545
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
ROC-AUC: 0.5

Confusion Matrix:
[[5 0]
 [6 0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.45      1.00      0.62         5
           1       0.00      0.00      0.00         6

    accuracy                           0.45        11
   macro avg       0.23      0.50      0.31        11
weighted avg       0.21      0.45      0.28        11



In [68]:
# Logistic Regression Model

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

logistic_model.fit(x_train, y_train)

evaluate_model(
    logistic_model,
    x_test,
    y_test,
    "LOGISTIC REGRESSION MODEL"
)


LOGISTIC REGRESSION MODEL
Accuracy: 0.5455
Precision: 0.6667
Recall: 0.3333
F1 Score: 0.4444
ROC-AUC: 0.5333

Confusion Matrix:
[[4 1]
 [4 2]]

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.80      0.62         5
           1       0.67      0.33      0.44         6

    accuracy                           0.55        11
   macro avg       0.58      0.57      0.53        11
weighted avg       0.59      0.55      0.52        11



In [69]:
# Comparing the Models

models = {
    "Dummy Classifier": baseline_model,
    "Logistic Regression": logistic_model
}

comparison_results = []

for model_name, model in models.items():

    predictions = model.predict(X_test)

    probabilities = model.predict_proba(X_test)[:, 1]

    comparison_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

comparison_df = pd.DataFrame(comparison_results)

print("\nMODEL COMPARISON")
print(comparison_df.round(4))


MODEL COMPARISON
                 Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
0     Dummy Classifier    0.4545     0.0000  0.0000    0.0000   0.5000
1  Logistic Regression    0.5455     0.6667  0.3333    0.4444   0.5333


In [72]:
# Logistic Regression Coefficients

fitted_preprocessor = logistic_model.named_steps["preprocessor"]

fitted_classifier = logistic_model.named_steps["classifier"]

coefficients = pd.DataFrame({
    "Feature": feature_columns,
    "Coefficient": fitted_classifier.coef_[0]
})

coefficients["AbsoluteCoefficient"] = (
    coefficients["Coefficient"].abs()
)

coefficients = coefficients.sort_values(
    by="AbsoluteCoefficient",
    ascending=False
)

print("\nLOGISTIC REGRESSION FEATURE COEFFICIENTS")
print(coefficients)


LOGISTIC REGRESSION FEATURE COEFFICIENTS
             Feature  Coefficient  AbsoluteCoefficient
0  AverageOrderValue    -0.243850             0.243850
1           Monetary     0.213011             0.213011
2      PurchaseTrend    -0.074496             0.074496


In [73]:
#Predict Churn Probability for Test Customers

churn_predictions = x_test.copy()

churn_predictions["Actual_Churn"] = y_test

churn_predictions["Predicted_Churn"] = (
    logistic_model.predict(X_test)
)

churn_predictions["Churn_Probability"] = (
    logistic_model.predict_proba(x_test)[:, 1]
)

churn_predictions["Churn_Risk"] = pd.cut(
    churn_predictions["Churn_Probability"],
    bins=[-0.001, 0.30, 0.70, 1.00],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

print("\nCUSTOMER CHURN PREDICTIONS")
print(churn_predictions.head(5))



CUSTOMER CHURN PREDICTIONS
    AverageOrderValue  Monetary  PurchaseTrend  Actual_Churn  Predicted_Churn  \
10             512.15    512.15           -1.0             0                0   
38             160.80    321.60            0.0             0                1   
18             342.29    684.58            0.0             0                0   
32             238.07    238.07            NaN             1                1   
33             822.52    822.52           -1.0             1                0   

    Churn_Probability   Churn_Risk  
10           0.450574  Medium Risk  
38           0.519933  Medium Risk  
18           0.497633  Medium Risk  
32           0.510091  Medium Risk  
33           0.384929  Medium Risk  


In [75]:
#Save model and outputs

OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

comparison_df.to_csv(
    OUTPUT_DIR / "model_comparison.csv",
    index=False
)

coefficients.to_csv(
    OUTPUT_DIR / "logistic_regression_coefficients.csv",
    index=False
)

churn_predictions.to_csv(
    OUTPUT_DIR / "customer_churn_predictions.csv",
    index=True
)

print("\nAll model outputs saved successfully!")


All model outputs saved successfully!
